In [2]:
from ovizioapi.capture import OvizioCapture
import matplotlib.pyplot as plt
import os
from ovizioapi import OvizioApiNet
OvizioApiNet.OvizioApiNet.set_UseGPU(True)

file_list = [
    
    "Z:/12_Clinical_Data/PC008/M1/Capture 1.h5",
    "Z:/12_Clinical_Data/PC008/M2/Capture 1.h5",
    "Z:/12_Clinical_Data/PC008/M3/Capture 1.h5",
    "Z:/12_Clinical_Data/PC009/M1/Capture 1.h5",
    "Z:/12_Clinical_Data/PC009/M2/Capture 1.h5",
    "Z:/12_Clinical_Data/PC009/M3/Capture 1.h5",
    
]

for file in file_list:
    cpt = OvizioCapture(str(file))
    num_images = len(cpt)

    # Extract the directory structure from the file path
    relative_dir = os.path.dirname(file).replace('Z:/12_Clinical_Data', 'Z:/12_Clinical_Data/Ovizio_Output')
    
    for i in range(num_images):
        # Make the output directory if it doesn't exist
        os.makedirs(relative_dir, exist_ok=True)

        # Save image in the specified directory
        plt.imsave(os.path.join(relative_dir, f'phase_img{i}.png'), cpt.get_phase(i), cmap='gray')



[ INIT ] Initialize Ovizio API...
[ INFO ] Using OsOne Path: C:\Program Files\Ovizio\OsOne\bin
[  OK  ] File information successfully extracted!
[ INFO ] OsOne Version: 5.12.13.24463
[ INIT ] Setting GPU usage to 'True'
[ INIT ] Checking Config Files
[  OK  ] Config Files Are Valid
[ INIT ] Initialization DONE!



In [ ]:
from ovizioapi.capture import OvizioCapture
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
import torch
import cv2
import numpy as np
import os
from PIL import Image


capture_path ="C:/Users/Admin/Desktop/CFP005-0/M1/Capture 1.h5"
cpt = OvizioCapture(capture_path)
num_frames = len(cpt)
reconstructed_imgs = []
output_dir = 'Z:/12_Clinical_Data/Ovizio_Output/test/'
cell_counts = {
        'RBC': 0,
        'WBC': 0,
        'PLT': 0
    }

for i in range(num_frames):
    phase_image = cpt.get_phase(i)
    reconstructed_imgs.append(phase_image)
    # new_phase_image = np.zeros([phase_image.shape[0], phase_image.shape[1], 3], dtype=np.float32)
    # new_phase_image[:,:,0], new_phase_image[:,:,1], new_phase_image[:,:,2] = np.asarray(phase_image), np.asarray(phase_image), np.asarray(phase_image)
    new_phase_image = np.dstack([phase_image, phase_image, phase_image])
    # norm_image = cv2.normalize(new_phase_image, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
    norm_image = ((new_phase_image - np.min(new_phase_image)) * (1/(np.max(new_phase_image) - np.min(new_phase_image)) * 255)).astype('uint8')
    detection_model = YOLO('C:/Users/Admin/Desktop/Projects/object_detect/v8xl_6_7_23_ovizio.pt')
    results = detection_model.predict(norm_image, classes=[0, 1, 2], 
                                      project=output_dir, name='', save=False, device='cuda:1')

    for result in results:
        boxes = result.boxes
        rbc_count = torch.sum(torch.eq(boxes.cls, 0.)).item()
        wbc_count = torch.sum(torch.eq(boxes.cls, 1.)).item()
        plt_count = torch.sum(torch.eq(boxes.cls, 2.)).item()
                
        cell_counts['RBC'] += rbc_count
        cell_counts['WBC'] += wbc_count  
        cell_counts['PLT'] += plt_count



print(cell_counts)

In [ ]:
from ovizioapi.capture import OvizioCapture
import numpy as np
from ultralytics import YOLO
import torch
import numpy as np
import cv2
import timeit
import matplotlib.pyplot as plt
from ovizioapi import OvizioApiNet
from PIL import Image
import sys
np.set_printoptions(threshold=sys.maxsize)

OvizioApiNet.OvizioApiNet.set_UseGPU(True)
print(f'UseGPU: {OvizioApiNet.OvizioApiNet.get_UseGPU()}')

capture_path ="C:/Users/Admin/Desktop/CFP007-0/M1/Capture 1.h5"
cpt = OvizioCapture(capture_path)
num_frames = len(cpt)
reconstructed_imgs = []
norm_image_list = []
output_dir = 'C:/Users/Admin/Desktop/CFP007-0/output'
cell_counts = {
        'RBC': 0,
        'WBC': 0,
        'PLT': 0
    }
test_png = []
test_raw = []
print (f'Total Images: {num_frames}')
start_time = timeit.default_timer()

for i in range(num_frames):
    phase_image = cpt.get_phase(i)
    # phase_image = cpt.get_phase(14)
    # plt.imsave("Z:/13_Siko/Test_data/CFP007-0(M1) npz data/Img_14.jpg", phase_image, cmap='gray')
    # cv2.imwrite('Z:/13_Siko/Test_data/CFP007-0(M1) npz data/Img_14.png', phase_image*255)
    # img_id = 14
    # png_pixel = np.array(cv2.imread('Z:/12_Clinical_Data/Ovizio_Output/CFP007-0/M1/phase_img{}.png'.format(img_id)))
    # png_pixel = np.array(cv2.imread('C:/Users/Admin/Desktop/Projects/ovizioapi/notebooks/phase_img{}.png'.format(0)))
    # png_pixel = np.dstack([png_pixel, png_pixel, png_pixel])
    # norm_png = cv2.normalize(png_pixel, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
    # test_png = norm_png
    # print(norm_png)
    reconstructed_imgs.append(phase_image)
elapsed = timeit.default_timer() - start_time
print (f'Reconstruction Time: {elapsed}')

start_time = timeit.default_timer()
for i in range(len(reconstructed_imgs)): 
   
    new_phase_image = np.dstack([reconstructed_imgs[i], reconstructed_imgs[i], reconstructed_imgs[i]])
    norm_image = cv2.normalize(new_phase_image, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
    # plt.imsave(f'phase_img{i}.png', norm_image, cmap='gray')
    # test_raw = norm_image
    # print(norm_image)
    # norm_image = ((new_phase_image - np.min(new_phase_image)) * (1/(np.max(new_phase_image) - np.min(new_phase_image)) * 255)).astype('uint8')
    norm_image_list.append(norm_image)
elapsed = timeit.default_timer() - start_time
print (f'Normalize Time: {elapsed}')

# start_time = timeit.default_timer()
# detection_model = YOLO('C:/Users/Admin/Desktop/Projects/object_detect/v8xl_6_7_23_ovizio.pt')
# batch_size = 300
# for i in range(0, len(norm_image_list), batch_size):
#     batch_images = norm_image_list[i:i+batch_size]
#     results = detection_model.predict(batch_images, classes=[0, 1, 2], 
#                                         project=output_dir, name='', save=False, device='cuda:0', show_labels=False, save_crop=False)

#     for result in results:
#         boxes = result.boxes
#         rbc_count = torch.sum(torch.eq(boxes.cls, 0.)).item()
#         wbc_count = torch.sum(torch.eq(boxes.cls, 1.)).item()
#         plt_count = torch.sum(torch.eq(boxes.cls, 2.)).item()
                
#         cell_counts['RBC'] += rbc_count
#         cell_counts['WBC'] += wbc_count  
#         cell_counts['PLT'] += plt_count
# elapsed = timeit.default_timer() - start_time
# print (f'Detection Time: {elapsed}')

# print(cell_counts)

In [ ]:

start_time = timeit.default_timer()
detection_model = YOLO('C:/Users/Admin/Desktop/Projects/object_detect/v8xl_6_7_23_ovizio.pt')
batch_size = 300
for i in range(0, len(norm_image_list), batch_size):
    batch_images = norm_image_list[i:i+batch_size]
    results = detection_model.predict(batch_images, classes=[0, 1, 2], 
                                        project=output_dir, name='', save=False, device='cuda:0', show_labels=False, save_crop=False,)

    for result in results:
        boxes = result.boxes
        rbc_count = torch.sum(torch.eq(boxes.cls, 0.)).item()
        wbc_count = torch.sum(torch.eq(boxes.cls, 1.)).item()
        plt_count = torch.sum(torch.eq(boxes.cls, 2.)).item()
                
        cell_counts['RBC'] += rbc_count
        cell_counts['WBC'] += wbc_count  
        cell_counts['PLT'] += plt_count
elapsed = timeit.default_timer() - start_time
print (f'Detection Time: {elapsed}')

print(cell_counts)

In [ ]:

torch.cuda.empty_cache()
